<p align="center">
  <strong>Run this notebook:</strong>
</p>

<p align="center">
  <a href="https://colab.research.google.com/github/MeteoSwiss/nwp-fdb-polytope-demo/blob/main/examples/earthkit/earthkit_onboarding_meteoswiss.ipynb">
    <img
      src="https://colab.research.google.com/assets/colab-badge.svg"
      alt="Open the main version in Google Colab"
    >
  </a>
  &nbsp;
  <a href="https://colab.research.google.com/github/MeteoSwiss/nwp-fdb-polytope-demo/blob/intro_earthkit_notebook/examples/earthkit/earthkit_onboarding_meteoswiss.ipynb">
    <img
      src="https://img.shields.io/badge/Open_dev_version-in%20Colab-F9AB00?logo=googlecolab&logoColor=white"
      alt="Open the development version in Google Colab"
    >
  </a>
</p>

<br>

<p align="center">
  <a href="https://earthkit.ecmwf.int/">
    <img
      src="https://github.com/ecmwf/logos/raw/refs/heads/main/logos/earthkit/earthkit-light.svg"
      alt="earthkit"
      width="520"
    >
  </a>
</p>

<h1 align="center">earthkit onboarding for MeteoSwiss NWP workflows</h1>

> **What is earthkit?**
>
> earthkit is an open-source Python ecosystem led by ECMWF. It provides a
> consistent workflow for accessing, inspecting, processing, analysing, and
> visualising weather and climate data—including the GRIB data commonly used
> in numerical weather prediction.

<p align="center">
  <a href="https://earthkit.ecmwf.int/">Website</a>
  &nbsp;·&nbsp;
  <a href="https://earthkit.readthedocs.io/en/latest/">Documentation</a>
  &nbsp;·&nbsp;
  <a href="https://github.com/ecmwf/earthkit">GitHub</a>
</p>

---

<p align="center">
  <img
    src="https://raw.githubusercontent.com/MeteoSwiss/nwp-fdb-polytope-demo/intro_earthkit_notebook/examples/earthkit/earthkit_components.png"
    alt="Overview of earthkit components"
    width="720"
  >
</p>

## earthkit components

| Package | Purpose |
|---|---|
| [`earthkit-data`](https://earthkit-data.readthedocs.io/en/stable/) | A format-agnostic Python interface for geospatial data, with a focus on meteorology and climate science. |
| [`earthkit-plots`](https://earthkit-plots.readthedocs.io/en/stable/) | Produce publication-quality weather and climate charts and maps with only a few lines of code. |
| [`earthkit-meteo`](https://earthkit-meteo.readthedocs.io/en/stable/) | Perform common meteorological calculations using NumPy, Torch, CuPy, xarray, or field lists. |
| [`earthkit-geo`](https://earthkit-geo.readthedocs.io/en/stable/) | Work with geospatial shapes, coordinates, grids, and projections. |
| [`earthkit-transforms`](https://earthkit-transforms.readthedocs.io/en/stable/) | Apply transformations, aggregations, and statistical analyses across data cubes. |
| [`earthkit-hydro`](https://earthkit-hydro.readthedocs.io/en/stable/) | Work with river networks, flow accumulation, and other hydrological data. |

---

## Table of contents

- [Why earthkit at MeteoSwiss?](#scrollTo=d5bd5057)
- [earthkit-data](#scrollTo=0502db9c)
  - [The mental model to remember](#scrollTo=118a08ee)
  - [Environment setup](#scrollTo=3e69327b)
    - [ecCodes and MeteoSwiss definitions](#scrollTo=312f7565)
  - [Read a MeteoSwiss forecast](#scrollTo=6b905d2a)
    - [FieldLists and fields](#scrollTo=Hcha38PeHF2f)
    - [Field metadata and geography](#scrollTo=y3L8-i8IOiBs)
    - [Modifying and selecting fields](#scrollTo=Hpr_e1l5jJlL)
  - [Converting to Xarray](#scrollTo=bEhzWbREk73t)
- [earthkit-geo](#scrollTo=e862b478)
  - [Regridding to a regular latitude–longitude grid](#scrollTo=b108cb7b)
- [earthkit-plots](#scrollTo=fcf2cc96)
  - [Plotting with earthkit-plots](#scrollTo=c03dad30)
- [Further learning and examples](#scrollTo=7e5e8344)


<a name="scrollTo=d5bd5057"></a>

## Why earthkit at MeteoSwiss?

MeteoSwiss uses gridded meteorological data across forecasting, research and machine-learning workflows. Many of these workflows repeat similar tasks, such as reading data, inspecting metadata, converting formats, regridding and plotting.

Using shared earthkit components can help to reduce duplicated MeteoSwiss-specific implementations and align with tools used across the meteorological community.

---

<a name="scrollTo=0502db9c"></a>

<a id="earthkit-data"></a>

![earthkit-data-logo](https://github.com/ecmwf/logos/raw/refs/heads/main/logos/earthkit/earthkit-data-light.svg)

<a name="scrollTo=118a08ee"></a>

## The mental model to remember

The most useful earthkit mental model is:

```text
source → earthkit data object → FieldList / Xarray / NumPy / Pandas → analysis or plotting
```

<a name="scrollTo=3e69327b"></a>

## Environment setup

In [ ]:
%pip install -q earthkit==1.0.0

<a name="scrollTo=312f7565"></a>

### ecCodes and MeteoSwiss definitions

GRIB files are decoded by [ecCodes](https://confluence.ecmwf.int/display/ECC). It provides the rules that translate the numeric identifiers stored in GRIB messages into metadata such as parameter names, units and level types.

Standard ecCodes definitions cover internationally defined parameters. Some MeteoSwiss and COSMO/ICON parameters require additional local definitions. These definitions must be made available **before** importing or reading GRIB data.

MeteoSwiss definitions: https://github.com/COSMO-ORG/eccodes-cosmo-resources/

The environment variable `ECCODES_DEFINITION_PATH` tells ecCodes where to find the additional definitions. The local directory should be prepended while retaining the default ecCodes search path.

> **Note:** This is a temporary workaround until the earthkit 1.x compatible definitions are publicly available through the `eccodes-cosmo-resources-python` package.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import os
import tarfile

ARCHIVE = Path("definitions.edzw-2.47.0-1.tar.bz2")

if not ARCHIVE.exists():
    urlretrieve(
        "https://raw.githubusercontent.com/"
        "MeteoSwiss/nwp-fdb-polytope-demo/"
        "intro_earthkit_notebook/examples/earthkit/"
        f"{ARCHIVE.name}",
        ARCHIVE,
    )

with tarfile.open(ARCHIVE, "r:bz2") as tar:
    tar.extractall(filter="data")

os.environ["ECCODES_DEFINITION_PATH"] = str(
    Path("definitions.edzw-2.47.0-1").resolve()
)
os.environ["ECCODES_VERSION_CHECK_OFF"] = "1"

> **Tip:** When changing `ECCODES_DEFINITION_PATH` in an existing notebook session, restart the kernel before reading GRIB data. ecCodes may already have loaded its definition paths.


<a name="scrollTo=6b905d2a"></a>

## Read a MeteoSwiss forecast

This example uses a small subset of an ICON-CH2-EPS forecast produced by MeteoSwiss:

- one forecast step;
- one variable: 2 m air temperature;
- the control member only.

The data is read directly from a URL. With `from_source()`, the source could also be a local file, FDB, Polytope or another supported input.

[See all supported sources in earthkit-data](https://earthkit-data.readthedocs.io/en/latest/concepts/inputs/from_source.html#from_source)

In [ ]:
import earthkit.data as ekd

data = ekd.from_source("sample", "icon_ch2_t2m.grib2")
data

The returned object provides some basic information but its primary goal is to convert the data into the required representation for further work. The actual data loading is deferred as much as possible, until the data is converted into a given type.

In [ ]:
print("Available conversions:", data.available_types)

<a name="scrollTo=Hcha38PeHF2f"></a>

### Fieldlists and fields
GRIB data can be converted into a [FieldList](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/fieldlist/index.html#earthkit.data.core.fieldlist.FieldList), which represents each GRIB message as a field. In earthkit a [field](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/field/index.html#earthkit.data.core.field.Field) is a horizontal slice of the atmosphere at a given time. In this sense the Field object is generic enough to represent other types of data than GRIB (e.g. NetCDF, GeoTIFF, dictionary data etc.)


In [ ]:
# fl is a fieldlist
fl = data.to_fieldlist()
print("Number of fields/messages:", len(fl))

[ls()](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/fieldlist/index.html#earthkit.data.core.fieldlist.FieldList.ls) lists the fields in the fieldlist.

In [ ]:
fl.ls()

In [ ]:
# the variable `field` is the first field in the fieldlist
field = fl[0]
print(field)

<a name="scrollTo=y3L8-i8IOiBs"></a>

#### Field metadata

Each `Field` exposes format-independent metadata grouped into logical components.

| Component | Common keys |
|---|---|
| **Parameter** | `parameter.variable`, `parameter.units`, `parameter.chem_variable` |
| **Time** | `time.base_datetime`, `time.valid_datetime`, `time.step` |
| **Vertical** | `vertical.level`, `vertical.level_type` |
| **Geography** | `geography.latitudes`, `geography.longitudes`, `geography.shape` |
| **Ensemble** | `ensemble.member` |

Metadata values can be accessed with the [`get()`](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/field/index.html#earthkit.data.core.field.Field.get) method.


In [ ]:
print("parameter.variable:  ", field.get("parameter.variable"))
print("parameter.units:  ", field.get("parameter.units"))
print("time.base_datetime:  ", field.get("time.base_datetime"))
print("...")

More examples about discovering metadata: [2026-earthkit-training earthkit-data notebook](https://github.com/ecmwf-training/2026-earthkit-training/blob/main/content/earthkit-data/ekd-2-grib.ipynb)

#### Field overview

To get an overview about the components/keys of a field simply use the automatic display.

In [ ]:
field

Among the metadata, the **geography** section contains information about the underlying model grid, including a unique grid identifier (`uid`), the grid name, its extent (`area`), and the number of grid cells (`shape`).


In [ ]:
field.geography.grid_spec()

For more information about the `ICON-CH2_C` grid from the ICON-CH2-EPS model, see the [Open Data documentation](https://opendatadocs.meteoswiss.ch/e-forecast-data/e2-e3-numerical-weather-forecasting-model).

#### Field geography

We can use [latlons()](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/field/component/geography/index.html#earthkit.data.field.component.geography.GeographyBase.latlons) to retrieve the latitude and longitude arrays for all grid cells.

This is especially useful for ICON data because ICON uses an unstructured triangular grid rather than a regular latitude–longitude grid. The grid points are therefore not arranged in simple rows and columns, which makes their coordinates less obvious. `latlons()` provides the geographic position of each cell directly.

In [ ]:
lat, lon = field.geography.latlons()
print("latitudes:", lat)
print("longitudes:", lon)

#### Field values


Use [to_numpy](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/field/index.html#earthkit.data.core.field.Field.to_numpy) to convert the field values into a NumPy array. By default, it preserves the field's grid dimensions, such as (rows, columns).

In [ ]:
a = field.to_numpy()
a.shape, a

<a name="scrollTo=Hpr_e1l5jJlL"></a>

## Modifying fields

Fields can be modified by [set()](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/field/index.html#earthkit.data.core.field.Field.set), which generates a new field with updated values/components.

In [ ]:
vals = field.values + 2.0
f_modified = field.set({"vertical.level": 1, "values": vals})
f_modified.ls()

In [ ]:
# compare the metadata in the old and new fields
field.get("vertical.level"), f_modified.get("vertical.level")

In [ ]:
# compare the values in the old and new fields
field.values.max(), f_modified.values.max()

### Field selection

Use [sel()](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/fieldlist/index.html#earthkit.data.core.fieldlist.FieldList.sel) to select the fields matching the given metadata conditions in a `fieldlist`.

In [ ]:
fl.sel({"parameter.variable": "T_2M"}).ls()

Examples about fields used in arithmetics: [2026-earthkit-training earthkit-data notebook](https://github.com/ecmwf-training/2026-earthkit-training/blob/main/content/earthkit-data/ekd-2-grib.ipynb)

<a name="scrollTo=bEhzWbREk73t"></a>

## Converting to Xarray

We can convert a `FieldList` to an Xarray `Dataset` using earthkit-data's own
[Xarray engine](https://earthkit-data.readthedocs.io/en/latest/concepts/xarray/overview.html).
For most use cases, the default conversion is sufficient.

In [ ]:
ds = data.to_xarray()
ds


### Using Xarray profiles

The Xarray engine supports [**profiles**](https://earthkit-data.readthedocs.io/en/latest/concepts/xarray/overview.html#profiles), which control how the resulting
`Dataset` is organised. A profile can change, for example, how dimensions are
named, how variables are grouped, or how metadata is exposed.

The default profile is suitable for most applications, but other profiles may
produce a layout that is more convenient for specific workflows.

In [ ]:
ds_grib = data.to_xarray(profile="grib")
ds_mars = data.to_xarray(profile="mars")


Compare the dimensions, coordinates and variables of the two datasets to see
how the profile changes the Xarray representation.


In [ ]:
print("MARS profile")
display(ds_mars)

In [ ]:
print("GRIB profile")
display(ds_grib)

The two profiles produce the same data layout for this ICON field. The main difference is in the dataset-level metadata: the `mars` profile adds MARS-specific attributes such as `levtype`, `date`, `time`, and `number`, while the `grib` profile keeps a more minimal set of attributes. The profile therefore changes how metadata is represented, not the underlying data values. One can also define a custom profile.

### `earthkit` accessor

Xarrays created with earthkit-data have the `earthkit` accessor. It is an experimental feature and for each DataArray it stores earthkit specific metadata.

In [ ]:
ds["T_2M"].earthkit.grid_spec

---

<a name="scrollTo=e862b478"></a>

<a id="earthkit-geo"></a>


![earthkit-geo-logo](https://github.com/ecmwf/logos/raw/refs/heads/main/logos/earthkit/earthkit-geo-light.svg)

<a name="scrollTo=b108cb7b"></a>

## Regridding to a regular latitude–longitude grid

ICON uses an unstructured triangular grid. This is well suited to numerical weather prediction, but many analysis and visualisation tools expect data on a regular grid, where values are arranged at fixed latitude and longitude intervals.

`earthkit-geo` can interpolate the ICON field onto such a regular latitude–longitude grid. Here we use a spacing of `0.05°` in both directions.


In [ ]:
import earthkit.geo as ekg

# Regrid from the unstructured ICON grid to a regular 0.05° lat/lon grid
field_ll = ekg.regrid(field, grid=[0.05, 0.05])
field_ll

The regridded field now has a regular two-dimensional structure. We can inspect its target grid and array shape:


In [ ]:
print("Grid specification:", field_ll.geography.grid_spec())
print("Data shape:", field_ll.to_numpy().shape)

---

<a name="scrollTo=fcf2cc96"></a>

<a id="earthkit-plots"></a>


![earthkit-plots-logo](https://github.com/ecmwf/logos/raw/refs/heads/main/logos/earthkit/earthkit-plots-light.svg)

<a name="scrollTo=c03dad30"></a>

## Plotting with earthkit-plots

`earthkit-plots` can plot the earthkit-data `Field` directly. It uses the field metadata to identify the coordinates, variable and units automatically.


### Quick plotting

For a first look at a field, `quickplot()` is the shortest option. It chooses a suitable plot style from the field metadata and adds the basic map elements automatically.


In [ ]:
import earthkit.plots as ekp

ekp.quickplot(field_ll)

### Customising the map

For more control over the output, create an `ekp.Map` explicitly. This allows you to define the map domain and add elements such as a legend, coastlines, borders, gridlines, and a metadata-based title.

In [ ]:

chart = ekp.Map(domain=[5.5, 10.8, 45.5, 48.2])
chart.plot(field_ll, units="celsius")
chart.legend()
chart.coastlines()
chart.borders()
chart.gridlines()
chart.title("ICON-CH2-EPS {variable_name} – {time:%Y-%m-%d %H:%M UTC}")
chart.show()

### Choosing the map domain

A common task is to focus on a particular region. The `domain` argument accepts a bounding box in the order:

```text
[west, east, south, north]
```

The following example focuses on the Alpine region while keeping the plotting code unchanged.


In [ ]:
alps = [5.0, 16.5, 43.5, 49.5]

chart = ekp.Map(domain=alps)
chart.plot(field_ll, units="celsius")
chart.legend()
chart.coastlines()
chart.borders()
chart.gridlines()
chart.title("2 m temperature over the Alpine region")
chart.show()

Because ICON uses an unstructured triangular grid, the values are not arranged in a regular latitude–longitude array. For a filled map, `earthkit-plots` interpolates the grid points onto the map projection for visualisation. Try it out by replacing the regridded data `field_ll` with the unstructured data `field` in the code snippet above.

### Saving a figure

The same chart can be saved for reports, presentations or automated workflows. The file format is inferred from the extension.


In [ ]:
chart.save("icon_ch2_temperature.png", dpi=150) # dpi defines the resolution of the image in dots per inch. If 'figure', use the figure's dpi value.

These examples cover three common plotting workflows:

- use `quickplot()` for rapid data inspection;
- create a `Map` when the domain and map elements should be controlled;
- use `save()` when the figure is needed outside the notebook.

The [earthkit-plots gallery](https://earthkit-plots.readthedocs.io/en/stable/examples/gallery/index.html) contains further examples for contours, wind, precipitation and other meteorological fields.


<a name="scrollTo=7e5e8344"></a>

## Further learning and examples

### Core documentation

- [earthkit](https://earthkit.readthedocs.io/en/latest/)
- [earthkit-data](https://earthkit-data.readthedocs.io/en/latest/)
- [earthkit-plots](https://earthkit-plots.readthedocs.io/en/latest/)
- [earthkit-geo](https://earthkit-geo.readthedocs.io/en/latest/)
- [earthkit-data 1.0 migration guide](https://earthkit-data.readthedocs.io/en/latest/release-notes/migration_1.0.0.html)

### Training and notebooks

- [ECMWF 2026 earthkit training](https://github.com/ecmwf-training/2026-earthkit-training/tree/main)
- [MeteoSwiss ICON-CH2 pollen forecast](https://github.com/MeteoSwiss/opendata-nwp-demos/blob/main/10_icon_ch2_pollen_forecast.ipynb)
- [MeteoSwiss Polytope polygon cut-out example](https://htmlpreview.github.io/?https://raw.githubusercontent.com/MeteoSwiss/nwp-fdb-polytope-demo/main/examples/snapshots/feature_polygon_country_cut-out.html)
